In [1]:
# ============================================================
# AI USE CASE 3: AI-GENERATED INSIGHTS AND NATURAL-LANGUAGE Q&A
# Step 1: compute a VERIFIED summary of the project (same rules as SQL,
#         Python EDA and Power BI) and save it as kpi_summary.json.
# Step 2: build insights_prompt.txt - paste it into Claude to get an
#         executive summary and to ask questions about the numbers.
# Step 3 (optional): if ANTHROPIC_API_KEY is set, call Claude directly.
# The AI only sees these summary tables, never row-level customer data.
# ============================================================
# pip install pandas numpy openpyxl      (optional: pip install anthropic)

import json
import os
from pathlib import Path

import numpy as np
import pandas as pd

WINDOW_START, WINDOW_END = pd.Timestamp("2015-01-01"), pd.Timestamp("2017-09-30")
FILE_PATH = Path("DataCo.xlsx")
MODEL = "claude-sonnet-5-5"     # used only in the optional API step

# ------------------------------------------------------------
# 1. LOAD AND CLEAN (same steps as the other scripts)
# ------------------------------------------------------------
df = pd.read_excel(FILE_PATH, engine="openpyxl")
df.columns = (df.columns.astype(str).str.strip().str.lower()
              .str.replace(r"[^a-z0-9]+", "_", regex=True).str.strip("_"))


def parse_mixed_date(value):
    """Text cells are M/D/YYYY; real Excel dates have day and month swapped."""
    if isinstance(value, str):
        return pd.to_datetime(value.strip(), format="%m/%d/%Y %H:%M")
    ts = pd.Timestamp(value)
    return pd.Timestamp(year=ts.year, month=ts.day, day=ts.month,
                        hour=ts.hour, minute=ts.minute)


df["order_ts"] = df["order_date_dateorders"].map(parse_mixed_date)
df["ship_ts"] = df["shipping_date_dateorders"].map(parse_mixed_date)
gap = (df["ship_ts"].dt.normalize() - df["order_ts"].dt.normalize()).dt.days
assert (gap == df["days_for_shipping_real"]).all(), "Date parsing is wrong"

w = df[df["order_ts"].dt.normalize().between(WINDOW_START, WINDOW_END)].copy()
w["year"] = w["order_ts"].dt.year
delivered = w[w["delivery_status"] != "Shipping canceled"].copy()
delivered["late"] = (delivered["late_delivery_risk"] == 1).astype(int)
delivered["hour"] = delivered["order_ts"].dt.hour


def table(frame, ndigits=2):
    return frame.round(ndigits).reset_index().to_dict(orient="records")


def group_summary(col):
    g = (w.groupby(col)
           .agg(orders=("order_id", "nunique"), net_sales=("order_item_total", "sum"),
                profit=("order_profit_per_order", "sum")))
    g["margin_pct"] = g["profit"] / g["net_sales"] * 100
    g["share_of_net_sales_pct"] = g["net_sales"] / g["net_sales"].sum() * 100
    return g.sort_values("net_sales", ascending=False)


# ------------------------------------------------------------
# 2. VERIFIED SUMMARY
# ------------------------------------------------------------
orders = w["order_id"].nunique()
net = w["order_item_total"].sum()
profit = w["order_profit_per_order"].sum()
loss = w[w["order_profit_per_order"] < 0]

cust_orders = w.groupby("customer_id")["order_id"].nunique()
first_year = w.groupby("customer_id")["order_ts"].min().dt.year.value_counts().sort_index()
country_net = w.groupby("order_country")["order_item_total"].sum().sort_values(ascending=False)

ship = (delivered.groupby("shipping_mode")
        .agg(delivered_items=("late", "size"), late_rate_pct=("late", "mean"),
             avg_scheduled_days=("days_for_shipment_scheduled", "mean"),
             avg_actual_days=("days_for_shipping_real", "mean")))
ship["late_rate_pct"] *= 100

same_day = delivered[delivered["shipping_mode"] == "Same Day"].copy()
same_day["ordered"] = np.where(same_day["hour"] < 12, "before noon", "noon or later")

segments = (w.groupby("customer_segment")
              .agg(customers=("customer_id", "nunique"), orders=("order_id", "nunique"),
                   net_sales=("order_item_total", "sum")))
segments["net_sales_per_customer"] = segments["net_sales"] / segments["customers"]
segments["late_rate_pct"] = delivered.groupby("customer_segment")["late"].mean() * 100

summary = {
    "scope": {
        "period": "2015-01-01 to 2017-09-30 (analysis window)",
        "grain_note": "orders = distinct order ids; items = order lines",
        "sales_definition": "net sales = order item total (after discount)",
        "late_rate_definition": "late items / delivered items; canceled shipments excluded",
    },
    "kpis": {
        "order_items": len(w), "orders": orders, "net_sales": round(net, 2),
        "total_profit": round(profit, 2), "profit_margin_pct": round(profit / net * 100, 2),
        "avg_order_value": round(net / orders, 2),
        "late_delivery_rate_pct": round(delivered["late"].mean() * 100, 2),
        "canceled_items_pct": round((len(w) - len(delivered)) / len(w) * 100, 2),
        "avg_scheduled_days": round(delivered["days_for_shipment_scheduled"].mean(), 2),
        "avg_actual_days": round(delivered["days_for_shipping_real"].mean(), 2),
    },
    "by_year": table(w.groupby("year").agg(orders=("order_id", "nunique"),
                                           net_sales=("order_item_total", "sum"))),
    "by_market": table(group_summary("market")),
    "top_10_categories": table(group_summary("category_name").head(10)),
    "top_5_regions": table(group_summary("order_region").head(5)),
    "top_5_countries_share_of_net_sales_pct": round(country_net.head(5).sum() / net * 100, 2),
    "losses": {
        "loss_making_items": len(loss),
        "loss_making_items_pct": round(len(loss) / len(w) * 100, 2),
        "loss_amount": round(loss["order_profit_per_order"].sum(), 2),
        "profit_before_losses": round(w.loc[w["order_profit_per_order"] > 0,
                                            "order_profit_per_order"].sum(), 2),
    },
    "shipping_modes": table(ship.sort_values("late_rate_pct", ascending=False)),
    "same_day_late_rate_by_order_time_pct": table(same_day.groupby("ordered")["late"].mean() * 100),
    "customers": {
        "customers": int(w["customer_id"].nunique()),
        "orders_per_customer": round(orders / w["customer_id"].nunique(), 2),
        "repeat_customers_pct": round((cust_orders >= 2).mean() * 100, 2),
        "new_customers_by_first_order_year": {int(k): int(v) for k, v in first_year.items()},
        "segments": table(segments),
    },
    "caveats": [
        "Data after 2017-09-30 has a different structure (about one item per order, much lower net sales) and is excluded.",
        "Each market sold mainly in separate blocks of months, so market totals reflect active months, not demand; do not compare markets by volume or growth.",
        "Losses (about 1 in 5 items) are equally frequent across discount levels, shipping modes, segments, markets and categories; no available field explains them. No cost data exists.",
        "Delivery fields look simulated (for example First Class always takes 2 days against a 1-day promise); treat delivery findings as indicative.",
        "No supplier or inventory data exists, so supplier and inventory questions cannot be answered.",
        "Department names do not map cleanly to categories.",
        "Customers are located only in the US and Puerto Rico while orders ship to five global markets.",
    ],
}

# Optional AI outputs from the earlier scripts, if the files are present.
for name, key in [("model_metrics.csv", "late_risk_model_test_results_2017"),
                  ("demand_forecast.csv", "baseline_forecast")]:
    path = Path(name)
    if path.exists():
        summary[key] = json.loads(pd.read_csv(path).round(4).to_json(orient="records",
                                                                     date_format="iso"))

Path("kpi_summary.json").write_text(json.dumps(summary, indent=2, default=str), encoding="utf-8")

# ------------------------------------------------------------
# 3. PROMPT FOR CLAUDE
# ------------------------------------------------------------
INSTRUCTIONS = """You are a senior supply chain analyst. Below is a VERIFIED summary of a
supply chain project (JSON). Rules:
1. Use ONLY the numbers in the JSON. Never invent, estimate or recalculate figures.
2. If a question cannot be answered from the JSON, say so and name the data that is missing.
3. Respect the "caveats": do not make claims they forbid (for example, no market growth
   or market share conclusions, and no explanation of why items lose money).
4. Write: (a) a 5-sentence executive summary, (b) 5 insights, each with its supporting
   number, (c) 5 business recommendations, each tied to an insight, (d) 3 risks or
   limitations of this analysis.
5. Plain business language for a non-technical manager."""

prompt = f"{INSTRUCTIONS}\n\n=== VERIFIED SUMMARY (JSON) ===\n{json.dumps(summary, indent=2, default=str)}\n"
Path("insights_prompt.txt").write_text(prompt, encoding="utf-8")

print("Verified KPIs:")
for k, v in summary["kpis"].items():
    print(f"  {k:26s} {v:,}")
print(f"\nSaved kpi_summary.json and insights_prompt.txt ({len(prompt):,} characters)")

# ------------------------------------------------------------
# 4. OPTIONAL: CALL CLAUDE DIRECTLY (needs ANTHROPIC_API_KEY and `pip install anthropic`)
# ------------------------------------------------------------
if os.environ.get("ANTHROPIC_API_KEY"):
    try:
        import anthropic
        reply = anthropic.Anthropic().messages.create(
            model=MODEL, max_tokens=1800,
            messages=[{"role": "user", "content": prompt}])
        Path("executive_summary.md").write_text(reply.content[0].text, encoding="utf-8")
        print("Saved executive_summary.md")
    except Exception as exc:                      # keep the script usable offline
        print("API step skipped:", exc)
else:
    print("No API key set: open insights_prompt.txt, copy everything, and paste it into Claude.")

Verified KPIs:
  order_items                171,962
  orders                     57,349
  net_sales                  30,774,233.92
  total_profit               3,685,985.03
  profit_margin_pct          11.98
  avg_order_value            536.61
  late_delivery_rate_pct     57.28
  canceled_items_pct         4.3
  avg_scheduled_days         2.93
  avg_actual_days            3.5

Saved kpi_summary.json and insights_prompt.txt (10,609 characters)
No API key set: open insights_prompt.txt, copy everything, and paste it into Claude.
